# Fine-Tune Coach Notebook

This notebook helps you run your first full loop:
1. Collect raw logs (`data/raw/`)
2. Build rewarded multi-step tool trajectories
3. Build preference pairs for RL-style training
4. Build SFT dataset and run MLX training

Everything is scoped to this repo and scripts under `fine-tune/`.

In [ ]:
from pathlib import Path
import json
import subprocess

ROOT = Path.cwd()
print('repo root:', ROOT)
print('fine-tune exists:', (ROOT / 'fine-tune').exists())

## 1) Collect raw data
Copies Claude and Anvil logs into `data/raw/`.

In [ ]:
subprocess.run(['bash', 'fine-tune/collect_raw_data.sh'], check=True)

## 2) Build rewarded trajectories from agentMemory DB
Set `project` to scope behavior, or `None` for global.

In [ ]:
project = str(Path.cwd())  # or None
cmd = [
    './.venv/bin/python', 'fine-tune/build_success_trajectories.py',
    '--project', project,
    '--limit', '12000',
    '--successful-only',
    '--profile', 'fine-tune/rl_reward_profile.json'
]
subprocess.run(cmd, check=True)

## 3) Inspect and tune reward profile
Edit `fine-tune/rl_reward_profile.json` and re-run step 2 to refine scoring.

In [ ]:
profile_path = Path('fine-tune/rl_reward_profile.json')
profile = json.loads(profile_path.read_text())
profile

## 4) Build preference pairs for DPO/ORPO style training
Pick latest scored episodes file from `data/processed/rl/`.

In [ ]:
scored_files = sorted(Path('data/processed/rl').glob('rl_episodes_scored_*.jsonl'))
latest_scored = scored_files[-1]
print('latest scored:', latest_scored)
subprocess.run([
    './.venv/bin/python', 'fine-tune/continuous_rl_loop.py',
    '--episodes', str(latest_scored),
    '--high-reward', '2.0',
    '--low-reward', '1.0'
], check=True)

## 5) Build SFT data (successful tool behavior)
Exports from DB then prepares train/val JSONL.

In [ ]:
subprocess.run([
    './.venv/bin/python', 'fine-tune/export_from_agent_memory.py',
    '--dataset-type', 'sft',
    '--project', str(Path.cwd()),
    '--limit', '5000',
    '--output-dir', 'data/raw/agent_memory'
], check=True)

sft_files = sorted(Path('data/raw/agent_memory').glob('sft_*.jsonl'))
latest_sft = sft_files[-1]
print('latest sft:', latest_sft)

subprocess.run([
    './.venv/bin/python', 'fine-tune/prepare_jsonl.py',
    '--input', str(latest_sft),
    '--input-format', 'agent_memory',
    '--output-dir', 'data/processed/fine_tune'
], check=True)

## 6) Download model to `/models/`
If model is gated/private, set `HUGGING_FACE_API` (or fallback `HF_TOKEN`) in your shell or notebook env.

In [ ]:
# Example:
# import os
# os.environ['HUGGING_FACE_API'] = 'hf_...'

subprocess.run([
    './.venv/bin/python', 'fine-tune/download_models.py',
    '--model', 'mlx-community/gemma-3-4b-it-4bit'
], check=False)

## 7) Generate + dry-run a training script
Matches your workflow goal for coached script generation.

In [ ]:
subprocess.run([
    './.venv/bin/python', 'fine-tune/generate_training_script.py',
    '--request', 'write a Python script using mlx-tune to fine-tune Gemma 4 E4B on a 16GB Mac',
    '--output', 'fine-tune/generated/train_gemma_16gb.py'
], check=True)

subprocess.run([
    './.venv/bin/python', 'fine-tune/train_mlx_tune.py'
], check=True)

## 8) If training fails, auto-diagnose logs
Uses rule-based issue detection (OOM, deps, auth, path, network).

In [ ]:
subprocess.run([
    './.venv/bin/python', 'fine-tune/debug_training_log.py',
    '--log', 'fine-tune/outputs/train.log'
], check=False)

## Coaching loop
For each iteration:
1. Re-score episodes (`build_success_trajectories.py`)
2. Check if high-reward episodes correlate with your expected quality
3. Tune reward weights/signals
4. Rebuild pairs + retrain
5. Evaluate on held-out prompts from your real workflow